In [ ]:
# Revision R1 - Cell 1: Matched-sample comparison (Reviewer 2 pt 2, Reviewer 3 pt 1)
# Every configuration is evaluated on the SAME 656 fused test samples.
# Network-only is run twice: (a) same training budget as fusion (3,061 rows),
# (b) its original full budget (201,396 rows). Per-seed predictions are saved
# so that the next cell can do paired bootstrap CIs and TOST without retraining.

from google.colab import drive
drive.mount('/content/drive')

import os, time, json
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report

BASE = '/content/drive/MyDrive/Research/EVSE_Fusion_Project'
JOURNAL_DIR = os.path.join(BASE, 'For Journal')
REV_DIR = os.path.join(JOURNAL_DIR, 'revision_R1')
os.makedirs(REV_DIR, exist_ok=True)

SEEDS_10 = [42, 123, 2026, 7, 99, 2024, 17, 88, 555, 777]
LABEL_ORDER = ['Benign', 'Recon', 'DoS']
label2id = {l: i for i, l in enumerate(LABEL_ORDER)}

def rf(seed):
    return RandomForestClassifier(n_estimators=200, max_depth=20,
                                  class_weight='balanced', random_state=seed, n_jobs=-1)

# ---------------- Load fused data ----------------
d = np.load(os.path.join(BASE, 'data/processed/fusion_paired_data.npz'))
Xtr, ytr, Xte, yte = d['X_train'], d['y_train'], d['X_test'], d['y_test']
nP, nH, nN = int(d['n_power_feats']), int(d['n_host_feats']), int(d['n_network_feats'])
sl = {'power': slice(0, nP), 'host': slice(nP, nP+nH), 'network': slice(nP+nH, nP+nH+nN)}
print(f"Fused train {Xtr.shape}, test {Xte.shape}; dims P/H/N = {nP}/{nH}/{nN}")
print("Test class counts:", {LABEL_ORDER[k]: int(v) for k, v in zip(*np.unique(yte, return_counts=True))})

# ---------------- Network split files (same feature logic as Cell 5) ----------------
NET_META = ['device', 'state', 'attack_raw', 'AttackGroup', 'source_file', 'n_flows_in_window']
def net_xy(df, cols=None):
    if cols is None:
        cols = [c for c in df.columns if c not in NET_META and not c.endswith('_std')]
        cols = df[cols].select_dtypes(include=[np.number]).columns.tolist()
    X = df[cols].apply(pd.to_numeric, errors='coerce').fillna(0).values
    return X, df['AttackGroup'].map(label2id).values, cols

net_tr = pd.read_csv(os.path.join(BASE, 'data/processed/splits/network_train.csv'), low_memory=False)
net_te = pd.read_csv(os.path.join(BASE, 'data/processed/splits/network_test.csv'), low_memory=False)
XNtr_full, yNtr_full, ncols = net_xy(net_tr)
XNte_full, _, _ = net_xy(net_te, ncols)
assert XNtr_full.shape[1] == nN, f"network feature count {XNtr_full.shape[1]} != fused {nN}"

# ---------------- Provenance check: which split did the fused NETWORK rows come from? ----------------
def rowset(X):
    return set(map(tuple, np.round(X.astype(float), 6)))
S_tr, S_te = rowset(XNtr_full), rowset(XNte_full)
def provenance(Xpart, name):
    rows = list(map(tuple, np.round(Xpart[:, sl['network']].astype(float), 6)))
    in_tr = sum(r in S_tr for r in rows); in_te = sum(r in S_te for r in rows)
    print(f"  fused {name}: {len(rows)} network rows -> found in network_TRAIN: {in_tr}, "
          f"in network_TEST: {in_te}, in neither: {len(rows) - sum((r in S_tr) or (r in S_te) for r in rows)}")
print("\nProvenance of the network block inside the fused data:")
provenance(Xtr, 'train'); provenance(Xte, 'test')

# ---------------- Run all configurations on the SAME 656 test samples ----------------
configs = ['network_matched_budget', 'network_full_budget', 'power_only', 'host_only',
           'early_fusion', 'late_fusion_uniform']
preds = {c: [] for c in configs}
probas = {c: [] for c in configs}
rows = []
t0 = time.time()
for seed in SEEDS_10:
    # (a) Network-only, same 3,061-row training budget as fusion
    m = rf(seed).fit(Xtr[:, sl['network']], ytr)
    probas['network_matched_budget'].append(m.predict_proba(Xte[:, sl['network']]))
    # (b) Network-only, original full training budget, evaluated on the fused test rows
    m = rf(seed).fit(XNtr_full, yNtr_full)
    probas['network_full_budget'].append(m.predict_proba(Xte[:, sl['network']]))
    # single-modality power / host on the same samples
    mp = rf(seed).fit(Xtr[:, sl['power']], ytr); pp = mp.predict_proba(Xte[:, sl['power']])
    mh = rf(seed).fit(Xtr[:, sl['host']], ytr);  ph = mh.predict_proba(Xte[:, sl['host']])
    probas['power_only'].append(pp); probas['host_only'].append(ph)
    # early fusion (all 947 features)
    m = rf(seed).fit(Xtr, ytr)
    probas['early_fusion'].append(m.predict_proba(Xte))
    # late fusion, uniform average (the configuration in the submitted paper)
    probas['late_fusion_uniform'].append((pp + ph + probas['network_matched_budget'][-1]) / 3.0)
    for c in configs:
        p = probas[c][-1].argmax(1); preds[c].append(p)
        rows.append({'config': c, 'seed': seed,
                     'macro_f1': f1_score(yte, p, average='macro'),
                     **{f'f1_{LABEL_ORDER[k]}': v for k, v in
                        enumerate(f1_score(yte, p, average=None, labels=[0, 1, 2]))}})
    print(f"  seed {seed} done ({time.time()-t0:.0f}s)")

res = pd.DataFrame(rows)
res.to_csv(os.path.join(REV_DIR, 'R1_cell1_matched_per_seed.csv'), index=False)
np.savez_compressed(os.path.join(REV_DIR, 'R1_cell1_predictions.npz'),
                    y_test=yte, seeds=np.array(SEEDS_10),
                    **{f'pred_{c}': np.array(preds[c]) for c in configs},
                    **{f'proba_{c}': np.array(probas[c]) for c in configs})

# ---------------- Summary ----------------
print("\n" + "="*70); print("MATCHED-SAMPLE RESULTS (656 identical test samples, 10 seeds)"); print("="*70)
summ = res.groupby('config')[['macro_f1', 'f1_Benign', 'f1_Recon', 'f1_DoS']].agg(['mean', 'std']).round(6)
print(summ.loc[configs])

print("\nPaired per-seed differences, fusion minus network_matched_budget:")
base = res[res.config == 'network_matched_budget'].sort_values('seed')['macro_f1'].values
for c in ['early_fusion', 'late_fusion_uniform', 'network_full_budget']:
    v = res[res.config == c].sort_values('seed')['macro_f1'].values
    dlt = v - base
    print(f"  {c:22s} mean={dlt.mean():+.6f}  min={dlt.min():+.6f}  max={dlt.max():+.6f}")

errs = {c: int((np.array(preds[c]) != yte).sum(1).mean()) for c in configs}
print("\nMean misclassified test samples per seed (out of 656):", errs)
print(f"\nTotal time: {(time.time()-t0)/60:.1f} min | Saved to {REV_DIR}")

Mounted at /content/drive
Fused train (3061, 947), test (656, 947); dims P/H/N = 4/903/40
Test class counts: {'Benign': 345, 'Recon': 172, 'DoS': 139}

Provenance of the network block inside the fused data:
  fused train: 3061 network rows -> found in network_TRAIN: 3061, in network_TEST: 762, in neither: 0
  fused test: 656 network rows -> found in network_TRAIN: 204, in network_TEST: 656, in neither: 0
  seed 42 done (21s)
  seed 123 done (42s)
  seed 2026 done (61s)
  seed 7 done (81s)
  seed 99 done (101s)
  seed 2024 done (121s)
  seed 17 done (140s)
  seed 88 done (162s)
  seed 555 done (181s)
  seed 777 done (202s)

MATCHED-SAMPLE RESULTS (656 identical test samples, 10 seeds)
                        macro_f1           f1_Benign            f1_Recon  \
                            mean       std      mean       std      mean   
config                                                                     
network_matched_budget  1.000000  0.000000  1.000000  0.000000  1.000000   
net

In [ ]:
# Revision R1 - Cell 2b: continuation of Cell 2 (fixes the pandas .query('sum') name clash)
# Relies on variables already in memory from Cell 2: net_te, dup, Xtr, ytr, Xte, yte, audit,
# REV_DIR, SEEDS_10, LABEL_ORDER.

import time
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

assert len(dup) == len(Xte), "Cell 2 variables missing - rerun Cell 2 from the top"

g = (pd.DataFrame({'file': net_te['source_file'].values, 'dup': dup})
     .groupby('file')['dup'].agg(rows='size', dup_rows='sum'))
top = g[g['dup_rows'] > 0].sort_values('dup_rows', ascending=False).head(10)
print("Test source files contributing duplicates (top 10):"); print(top)
print(f"Test files with >=1 duplicate: {(g['dup_rows'] > 0).sum()} of {len(g)}")

# Unique-vector view: how many DISTINCT test vectors does each class actually contain?
kte_s = pd.Series(list(map(tuple, np.round(Xte.astype(float), 6))))
uniq = pd.DataFrame({'k': kte_s, 'c': [LABEL_ORDER[i] for i in yte], 'dup': dup})
print("\nDistinct test vectors per class (all / not seen in train):")
print(uniq.groupby('c').agg(distinct_all=('k', 'nunique'),
                             distinct_unseen=('k', lambda s: s[~uniq.loc[s.index, 'dup']].nunique())))

# ---------------- Evaluation: all test rows vs duplicate-free test rows ----------------
nd = ~dup
bin_all = yte != 0; bin_nd = (yte != 0) & nd
rows = []; t0 = time.time()
for seed in SEEDS_10:
    m = RandomForestClassifier(n_estimators=200, max_depth=20, class_weight='balanced',
                               random_state=seed, n_jobs=-1).fit(Xtr, ytr)
    p = m.predict(Xte)
    r = {'seed': seed,
         'f1_3class_all': f1_score(yte, p, average='macro', labels=[0, 1, 2]),
         'f1_3class_dupfree': f1_score(yte[nd], p[nd], average='macro', labels=[0, 1, 2]),
         'f1_reconDoS_all': f1_score(yte[bin_all], p[bin_all], average='macro', labels=[1, 2]),
         'f1_reconDoS_dupfree': f1_score(yte[bin_nd], p[bin_nd], average='macro', labels=[1, 2]),
         'errors_all': int((p != yte).sum()), 'errors_dupfree': int((p[nd] != yte[nd]).sum()),
         'n_dupfree': int(nd.sum()),
         'benign_errors_all': int(((yte == 0) & (p != yte)).sum()),
         'recon_dupfree_errors': int(((yte == 1) & nd & (p != yte)).sum()),
         'dos_dupfree_errors': int(((yte == 2) & nd & (p != yte)).sum())}
    for k, lab in enumerate(LABEL_ORDER):
        r[f'f1_{lab}_all'] = f1_score(yte, p, average=None, labels=[0, 1, 2])[k]
        r[f'f1_{lab}_dupfree'] = f1_score(yte[nd], p[nd], average=None, labels=[0, 1, 2])[k]
    rows.append(r); print(f"  seed {seed} done ({time.time()-t0:.0f}s)")

res = pd.DataFrame(rows)
res.to_csv(os.path.join(REV_DIR, 'R1_cell2_dupfree_eval.csv'), index=False)
audit.to_csv(os.path.join(REV_DIR, 'R1_cell2_duplicate_audit.csv'))
g.to_csv(os.path.join(REV_DIR, 'R1_cell2_duplicates_by_file.csv'))

print("\n" + "="*70); print("NETWORK-ONLY RF, 10 seeds: all test rows vs duplicate-free test rows"); print("="*70)
print(res.drop(columns='seed').agg(['mean', 'std', 'min', 'max']).T.round(6))
print(f"\nTotal time: {(time.time()-t0)/60:.1f} min | Saved to {REV_DIR}")

Test source files contributing duplicates (top 10):
                                          rows  dup_rows
file                                                    
EVSE-B-MaliciousEV-syn-stealth-scan.csv  22800     22790
EVSE-B-charging-os-fingerprinting.csv     6797      6549
EVSE-B-idle-os-fingerprinting.csv         4566      4229
EVSE-A-charging-Aggressive-scan.csv        999       686
EVSE-A-idle-portscan.csv                   497       384
EVSE-A-charging-os-fingerprinting.csv      476       325
EVSE-B-idle-icmp-flood.csv                   3         3
EVSE-A-Charging-Benign.csv                   4         1
Test files with >=1 duplicate: 8 of 11

Distinct test vectors per class (all / not seen in train):
        distinct_all  distinct_unseen
c                                    
Benign            13               12
DoS            13111            13108
Recon           2622             1137
  seed 42 done (18s)
  seed 123 done (34s)
  seed 2026 done (49s)
  seed 7 done (65s)
  s

In [ ]:
# Revision R1 - Cell 3: Part 2 (Power+Host vs Host-only, 4-class) - equivalence testing
# Reviewer 2 pts 3-4, Reviewer 3 pt 3.
# Design: 5 stratified re-partitions x 5 pairing seeds x 5 model seeds = 125 runs.
# PRE-SPECIFIED before any result (12 Nov 2026): equivalence margin DELTA = 0.005 macro-F1
# (primary) and 0.010 (sensitivity). Equivalence is claimed only if the 90% CI of the
# mean paired difference lies inside (-DELTA, +DELTA) AND both TOST p-values (Holm) < 0.05.

import os, time
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from scipy import stats

BASE = '/content/drive/MyDrive/Research/EVSE_Fusion_Project'
REV_DIR = os.path.join(BASE, 'For Journal', 'revision_R1')
CLASSES = ['Benign', 'Recon', 'DoS', 'Cryptojacking']; lab = {c: i for i, c in enumerate(CLASSES)}
DELTA_PRIMARY, DELTA_SENS = 0.005, 0.010
SPLIT_SEEDS = [11, 22, 33, 44, 55]          # new partitions (the paper used one fixed split)
PAIR_SEEDS = [42, 101, 202, 303, 404]       # 42 = the paper's pairing seed
MODEL_SEEDS = [42, 123, 2026, 7, 99]
B = 500                                     # test-sample bootstrap resamples per run

META_POWER = ['time', 'State', 'Attack', 'Attack-Group', 'Label', 'interface', 'AttackGroup']
META_HOST = ['State', 'Attack', 'Scenario', 'Label', 'interface', 'AttackGroup', 'time', 'msec', 'unit']

power = pd.read_csv(os.path.join(BASE, 'data/processed/power_clean.csv'), low_memory=False)
host = pd.read_csv(os.path.join(BASE, 'data/processed/host_clean.csv'), low_memory=False)
power = power[power['AttackGroup'].isin(CLASSES)].reset_index(drop=True)
host = host[host['AttackGroup'].isin(CLASSES)].reset_index(drop=True)
pcols = power[[c for c in power.columns if c not in META_POWER]].select_dtypes(include=[np.number]).columns.tolist()
hcols = host[[c for c in host.columns if c not in META_HOST]].select_dtypes(include=[np.number]).columns.tolist()
assert len(pcols) == 4 and len(hcols) == 903, (len(pcols), len(hcols))
PX = power[pcols].apply(pd.to_numeric, errors='coerce').fillna(0).values; Py = power['AttackGroup'].map(lab).values
HX = host[hcols].apply(pd.to_numeric, errors='coerce').fillna(0).values;  Hy = host['AttackGroup'].map(lab).values

# ---- Probe: grouping columns available for a capture-grouped split (used in the next cell) ----
print("Host candidate grouping columns:")
for c in ['Scenario', 'State', 'Attack', 'interface']:
    if c in host.columns:
        print(f"  {c}: {host[c].nunique()} unique | per class: "
              f"{host.groupby('AttackGroup')[c].nunique().to_dict()}")
print("Power candidate grouping columns:")
for c in ['State', 'Attack', 'interface']:
    if c in power.columns:
        print(f"  {c}: {power[c].nunique()} unique | per class: "
              f"{power.groupby('AttackGroup')[c].nunique().to_dict()}")

def split_idx(y, seed):   # 70/15/15 stratified, same protocol as the paper
    idx = np.arange(len(y))
    tr, tmp = train_test_split(idx, test_size=0.30, stratify=y, random_state=seed)
    va, te = train_test_split(tmp, test_size=0.50, stratify=y[tmp], random_state=seed)
    return tr, va, te

def pair(h_idx, p_idx, rng):   # host-anchored, class-conditional, power sampled with replacement
    Xs, ys = [], []
    for k in range(len(CLASSES)):
        h = h_idx[Hy[h_idx] == k]; pool = p_idx[Py[p_idx] == k]
        pp = rng.choice(pool, size=len(h), replace=True)
        Xs.append(np.hstack([PX[pp], HX[h]])); ys.append(np.full(len(h), k))
    return np.vstack(Xs), np.concatenate(ys)

def rf(s): return RandomForestClassifier(n_estimators=200, max_depth=20, class_weight='balanced',
                                         random_state=s, n_jobs=-1)
mf1 = lambda y, p: f1_score(y, p, average='macro', labels=[0, 1, 2, 3])

rows = []; t0 = time.time()
for ss in SPLIT_SEEDS:
    htr, _, hte = split_idx(Hy, ss); ptr, _, pte = split_idx(Py, ss)
    for ps in PAIR_SEEDS:
        rng = np.random.default_rng(ps)
        Xtr, ytr = pair(htr, ptr, rng); Xte, yte = pair(hte, pte, rng)
        for ms in MODEL_SEEDS:
            mh = rf(ms).fit(Xtr[:, 4:], ytr); ph = mh.predict_proba(Xte[:, 4:])
            me = rf(ms).fit(Xtr, ytr);       pe = me.predict(Xte)
            mp = rf(ms).fit(Xtr[:, :4], ytr); pl = ((mp.predict_proba(Xte[:, :4]) + ph) / 2).argmax(1)
            yh = ph.argmax(1)
            f_h, f_e, f_l = mf1(yte, yh), mf1(yte, pe), mf1(yte, pl)
            # paired test-sample bootstrap of the differences (same resampled rows for both arms)
            brng = np.random.default_rng(ms * 1000 + ps)
            de, dl = np.empty(B), np.empty(B)
            for b in range(B):
                i = brng.integers(0, len(yte), len(yte))
                fh = mf1(yte[i], yh[i]); de[b] = mf1(yte[i], pe[i]) - fh; dl[b] = mf1(yte[i], pl[i]) - fh
            rows.append({'split_seed': ss, 'pair_seed': ps, 'model_seed': ms, 'n_test': len(yte),
                         'f1_host': f_h, 'f1_early': f_e, 'f1_late': f_l,
                         'd_early': f_e - f_h, 'd_late': f_l - f_h,
                         'd_early_lo95': np.percentile(de, 2.5), 'd_early_hi95': np.percentile(de, 97.5),
                         'd_late_lo95': np.percentile(dl, 2.5), 'd_late_hi95': np.percentile(dl, 97.5),
                         'err_host': int((yh != yte).sum()), 'err_early': int((pe != yte).sum()),
                         'err_late': int((pl != yte).sum())})
        print(f"  split {ss} pair {ps} done ({time.time()-t0:.0f}s)")

R = pd.DataFrame(rows); R.to_csv(os.path.join(REV_DIR, 'R1_cell3_part2_runs.csv'), index=False)

# ---------------- Equivalence analysis ----------------
# Unit of analysis = (partition, pairing) cell, averaged over model seeds -> 25 units.
U = R.groupby(['split_seed', 'pair_seed'])[['d_early', 'd_late', 'f1_host', 'f1_early', 'f1_late']].mean().reset_index()
def tost(d, delta):
    n = len(d); m = d.mean(); se = d.std(ddof=1) / np.sqrt(n)
    p_lo = 1 - stats.t.cdf((m + delta) / se, n - 1)     # H0: mean <= -delta
    p_hi = stats.t.cdf((m - delta) / se, n - 1)         # H0: mean >= +delta
    ci90 = stats.t.interval(0.90, n - 1, loc=m, scale=se)
    return m, se, ci90, max(p_lo, p_hi)

def cluster_boot_ci(df, col, reps=5000, seed=0):    # resample partitions, then units within them
    r = np.random.default_rng(seed); parts = df['split_seed'].unique(); out = []
    for _ in range(reps):
        sel = r.choice(parts, len(parts), replace=True)
        vals = [df[df.split_seed == s][col].sample(frac=1, replace=True, random_state=int(r.integers(1e9))).mean()
                for s in sel]
        out.append(np.mean(vals))
    return np.percentile(out, [5, 95])

print("\n" + "="*70); print("PART 2 EQUIVALENCE (25 partition x pairing units, 5 model seeds each)"); print("="*70)
print(U[['f1_host', 'f1_early', 'f1_late']].agg(['mean', 'std', 'min', 'max']).round(6))
summary = []
for arm in ['d_early', 'd_late']:
    for delta in [DELTA_PRIMARY, DELTA_SENS]:
        m, se, ci, p = tost(U[arm].values, delta)
        summary.append({'comparison': arm.replace('d_', '') + '_minus_host', 'delta': delta,
                        'mean_diff': m, 'se': se, 'ci90_lo': ci[0], 'ci90_hi': ci[1], 'p_tost': p})
S = pd.DataFrame(summary)
for delta in [DELTA_PRIMARY, DELTA_SENS]:            # Holm across the two comparisons per margin
    idx = S.index[S.delta == delta]; order = S.loc[idx, 'p_tost'].sort_values().index
    for rank, i in enumerate(order):
        S.loc[i, 'p_tost_holm'] = min(1.0, S.loc[i, 'p_tost'] * (len(order) - rank))
    S.loc[idx, 'p_tost_holm'] = S.loc[idx, 'p_tost_holm'].cummax() if False else S.loc[idx, 'p_tost_holm']
for arm in ['d_early', 'd_late']:
    lo, hi = cluster_boot_ci(U, arm)
    S.loc[S.comparison == arm.replace('d_', '') + '_minus_host', 'cluster_boot90_lo'] = lo
    S.loc[S.comparison == arm.replace('d_', '') + '_minus_host', 'cluster_boot90_hi'] = hi
S['equivalent'] = ((S.ci90_lo > -S.delta) & (S.ci90_hi < S.delta) &
                   (S.cluster_boot90_lo > -S.delta) & (S.cluster_boot90_hi < S.delta) & (S.p_tost_holm < 0.05))
S.to_csv(os.path.join(REV_DIR, 'R1_cell3_part2_equivalence.csv'), index=False)
print(S.round(6).to_string(index=False))

print("\nPer-run test-sample bootstrap: share of runs whose 95% CI for the difference excludes 0:")
for arm in ['early', 'late']:
    ex = ((R[f'd_{arm}_lo95'] > 0) | (R[f'd_{arm}_hi95'] < 0)).mean()
    print(f"  {arm}: {ex:.1%} of {len(R)} runs")
print(f"\nTotal time: {(time.time()-t0)/60:.1f} min | Saved to {REV_DIR}")

Host candidate grouping columns:
  Scenario: 4 unique | per class: {'Benign': 1, 'Cryptojacking': 1, 'DoS': 2, 'Recon': 1}
  State: 2 unique | per class: {'Benign': 2, 'Cryptojacking': 2, 'DoS': 2, 'Recon': 2}
  Attack: 18 unique | per class: {'Benign': 1, 'Cryptojacking': 1, 'DoS': 8, 'Recon': 8}
  interface: 5 unique | per class: {'Benign': 1, 'Cryptojacking': 2, 'DoS': 1, 'Recon': 3}
Power candidate grouping columns:
  State: 2 unique | per class: {'Benign': 2, 'Cryptojacking': 2, 'DoS': 1, 'Recon': 1}
  Attack: 6 unique | per class: {'Benign': 1, 'Cryptojacking': 1, 'DoS': 2, 'Recon': 2}
  interface: 3 unique | per class: {'Benign': 1, 'Cryptojacking': 1, 'DoS': 1, 'Recon': 1}
  split 11 pair 42 done (51s)
  split 11 pair 101 done (101s)
  split 11 pair 202 done (150s)
  split 11 pair 303 done (197s)
  split 11 pair 404 done (242s)
  split 22 pair 42 done (287s)
  split 22 pair 101 done (331s)
  split 22 pair 202 done (376s)
  split 22 pair 303 done (420s)
  split 22 pair 404 done 

In [ ]:
# Revision R1 - Cell 4: Stronger fusion baselines + degraded/missing network (Reviewer 2 pt 7)
# Part 1 (Benign/Recon/DoS). Pairing rebuilt from the paper's fixed splits with 5 pairing seeds
# (train<-train pools, val<-val pools, test<-test pools); 5 model seeds -> 25 runs per condition.
# Degradation is applied to the NETWORK block at TEST time only (deployment fault model).

import os, time, warnings
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import f1_score
warnings.filterwarnings('ignore')

BASE = '/content/drive/MyDrive/Research/EVSE_Fusion_Project'
REV_DIR = os.path.join(BASE, 'For Journal', 'revision_R1')
SPLITS = os.path.join(BASE, 'data/processed/splits')
CLASSES = ['Benign', 'Recon', 'DoS']; lab = {c: i for i, c in enumerate(CLASSES)}
PAIR_SEEDS = [42, 101, 202, 303, 404]; MODEL_SEEDS = [42, 123, 2026, 7, 99]
NOISE = [0.5, 1.0, 2.0]

META = {'power': ['time', 'State', 'Attack', 'Attack-Group', 'Label', 'interface', 'AttackGroup'],
        'host': ['State', 'Attack', 'Scenario', 'Label', 'interface', 'AttackGroup', 'time', 'msec', 'unit'],
        'network': ['device', 'state', 'attack_raw', 'AttackGroup', 'source_file', 'n_flows_in_window']}
def load(mod, part):
    df = pd.read_csv(os.path.join(SPLITS, f'{mod}_{part}.csv'), low_memory=False)
    df = df[df['AttackGroup'].isin(CLASSES)]
    cols = [c for c in df.columns if c not in META[mod] and not (mod == 'network' and c.endswith('_std'))]
    cols = df[cols].select_dtypes(include=[np.number]).columns.tolist()
    return df[cols].apply(pd.to_numeric, errors='coerce').fillna(0).values, df['AttackGroup'].map(lab).values
D = {m: {p: load(m, p) for p in ['train', 'val', 'test']} for m in ['power', 'host', 'network']}
nP, nH, nN = D['power']['train'][0].shape[1], D['host']['train'][0].shape[1], D['network']['train'][0].shape[1]
assert (nP, nH, nN) == (4, 903, 40), (nP, nH, nN)
SP, SH, SN = slice(0, nP), slice(nP, nP + nH), slice(nP + nH, nP + nH + nN)

def pair(part, rng):   # host-anchored, class-conditional, power/network drawn from the SAME partition
    HX, Hy = D['host'][part]; PX, Py = D['power'][part]; NX, Ny = D['network'][part]
    Xs, ys = [], []
    for k in range(3):
        h = np.where(Hy == k)[0]
        p = rng.choice(np.where(Py == k)[0], len(h), replace=True)
        n = rng.choice(np.where(Ny == k)[0], len(h), replace=True)
        Xs.append(np.hstack([PX[p], HX[h], NX[n]])); ys.append(np.full(len(h), k))
    return np.vstack(Xs), np.concatenate(ys)

def rf(s): return RandomForestClassifier(n_estimators=200, max_depth=20, class_weight='balanced',
                                         random_state=s, n_jobs=-1)
def calib(m, X, y):
    try:
        from sklearn.frozen import FrozenEstimator
        return CalibratedClassifierCV(FrozenEstimator(m), method='isotonic').fit(X, y)
    except ImportError:
        return CalibratedClassifierCV(m, method='isotonic', cv='prefit').fit(X, y)
mf1 = lambda y, p: f1_score(y, p, average='macro', labels=[0, 1, 2])
GRID = [(a / 10, b / 10, (10 - a - b) / 10) for a in range(11) for b in range(11 - a)]  # weights P,H,N

rows = []; t0 = time.time()
for ps in PAIR_SEEDS:
    rng = np.random.default_rng(ps)
    Xtr, ytr = pair('train', rng); Xva, yva = pair('val', rng); Xte, yte = pair('test', rng)
    mu, sd = Xtr[:, SN].mean(0), Xtr[:, SN].std(0) + 1e-12
    for ms in MODEL_SEEDS:
        M = {k: rf(ms).fit(Xtr[:, s], ytr) for k, s in [('P', SP), ('H', SH), ('N', SN)]}
        ME = rf(ms).fit(Xtr, ytr)
        C = {k: calib(M[k], Xva[:, s], yva) for k, s in [('P', SP), ('H', SH), ('N', SN)]}
        Pva = {k: M[k].predict_proba(Xva[:, s]) for k, s in [('P', SP), ('H', SH), ('N', SN)]}
        w_best = max(GRID, key=lambda w: mf1(yva, (w[0]*Pva['P'] + w[1]*Pva['H'] + w[2]*Pva['N']).argmax(1)))
        stack = LogisticRegression(max_iter=3000).fit(np.hstack([Pva['P'], Pva['H'], Pva['N']]), yva)

        conds = {'clean': Xte[:, SN]}
        nrng = np.random.default_rng(ps * 7 + ms)
        for s_ in NOISE:
            conds[f'noise_{s_}sd'] = Xte[:, SN] + nrng.normal(0, 1, Xte[:, SN].shape) * sd * s_
        conds['network_missing'] = np.tile(mu, (len(yte), 1))     # missing -> train-mean imputation

        for cname, XN in conds.items():
            Xc = Xte.copy(); Xc[:, SN] = XN
            pr = {k: M[k].predict_proba(Xc[:, s]) for k, s in [('P', SP), ('H', SH), ('N', SN)]}
            cp = {k: C[k].predict_proba(Xc[:, s]) for k, s in [('P', SP), ('H', SH), ('N', SN)]}
            preds = {
                'network_only': pr['N'].argmax(1), 'host_only': pr['H'].argmax(1), 'power_only': pr['P'].argmax(1),
                'early_fusion': ME.predict(Xc),
                'late_uniform': ((pr['P'] + pr['H'] + pr['N']) / 3).argmax(1),
                'late_val_weighted': (w_best[0]*pr['P'] + w_best[1]*pr['H'] + w_best[2]*pr['N']).argmax(1),
                'stacking_lr': stack.predict(np.hstack([pr['P'], pr['H'], pr['N']])),
                'late_calibrated': ((cp['P'] + cp['H'] + cp['N']) / 3).argmax(1),
                'gate_maxconf': np.stack([cp['P'], cp['H'], cp['N']])[
                    np.stack([cp['P'], cp['H'], cp['N']]).max(2).argmax(0), np.arange(len(yte))].argmax(1),
            }
            if cname == 'network_missing':       # availability-aware: drop the missing channel
                preds['late_available_PH'] = ((cp['P'] + cp['H']) / 2).argmax(1)
            for arm, p in preds.items():
                rows.append({'pair_seed': ps, 'model_seed': ms, 'condition': cname, 'arm': arm,
                             'macro_f1': mf1(yte, p), 'errors': int((p != yte).sum()),
                             'benign_fpr': float(((yte == 0) & (p != 0)).sum() / max(1, (yte == 0).sum())),
                             'w_P': w_best[0], 'w_H': w_best[1], 'w_N': w_best[2]})
    print(f"  pairing seed {ps} done ({time.time()-t0:.0f}s) | val-tuned weights (P,H,N) last seed: {w_best}")

R = pd.DataFrame(rows); R.to_csv(os.path.join(REV_DIR, 'R1_cell4_fusion_degradation_runs.csv'), index=False)

ORDER = ['network_only', 'host_only', 'power_only', 'early_fusion', 'late_uniform', 'late_val_weighted',
         'stacking_lr', 'late_calibrated', 'gate_maxconf', 'late_available_PH']
COND = ['clean'] + [f'noise_{s}sd' for s in NOISE] + ['network_missing']
tab = R.pivot_table(index='arm', columns='condition', values='macro_f1', aggfunc='mean').reindex(ORDER)[COND]
print("\n" + "="*70); print("MEAN MACRO-F1 (25 runs per cell)"); print("="*70); print(tab.round(4).to_string())

print("\nPaired difference vs the best single channel AVAILABLE in that condition (mean [min, max] over 25 runs):")
for c in COND:
    ref = 'host_only' if c == 'network_missing' else 'network_only'
    base = R[(R.condition == c) & (R.arm == ref)].set_index(['pair_seed', 'model_seed'])['macro_f1']
    print(f"\n[{c}] reference = {ref}")
    for a in ORDER:
        sub = R[(R.condition == c) & (R.arm == a)]
        if sub.empty or a == ref: continue
        d = sub.set_index(['pair_seed', 'model_seed'])['macro_f1'] - base
        print(f"  {a:18s} {d.mean():+.5f}  [{d.min():+.5f}, {d.max():+.5f}]")
print(f"\nTotal time: {(time.time()-t0)/60:.1f} min | Saved to {REV_DIR}")

  pairing seed 42 done (43s) | val-tuned weights (P,H,N) last seed: (0.0, 0.0, 1.0)
  pairing seed 101 done (85s) | val-tuned weights (P,H,N) last seed: (0.0, 0.5, 0.5)
  pairing seed 202 done (127s) | val-tuned weights (P,H,N) last seed: (0.0, 0.5, 0.5)
  pairing seed 303 done (169s) | val-tuned weights (P,H,N) last seed: (0.0, 0.2, 0.8)
  pairing seed 404 done (211s) | val-tuned weights (P,H,N) last seed: (0.0, 0.4, 0.6)

MEAN MACRO-F1 (25 runs per cell)
condition           clean  noise_0.5sd  noise_1.0sd  noise_2.0sd  network_missing
arm                                                                              
network_only       0.9997       0.8345       0.6520       0.5330           0.2162
host_only          0.9877       0.9877       0.9877       0.9877           0.9877
power_only         0.7046       0.7046       0.7046       0.7046           0.7046
early_fusion       1.0000       0.9848       0.9599       0.9040           0.5901
late_uniform       0.9982       0.9904       0.

In [ ]:
# Revision R1 - Cell 5
# (A) Paired statistics for Cell 4 (no retraining)        - Reviewer 2 pt 7
# (B) Grouped feature removal: leave-one-group-out + greedy cumulative removal - Reviewer 2 pt 6
#     Protocol for (B): train on UNIQUE training vectors, test on duplicate-free test rows (Cell 2),
#     5 model seeds, Recon/DoS and 3-class macro-F1 reported.

import os, time, re
import numpy as np, pandas as pd
from scipy import stats
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

BASE = '/content/drive/MyDrive/Research/EVSE_Fusion_Project'
REV_DIR = os.path.join(BASE, 'For Journal', 'revision_R1')
SPLITS = os.path.join(BASE, 'data/processed/splits')

# ======================= (A) Statistics on Cell 4 =======================
R = pd.read_csv(os.path.join(REV_DIR, 'R1_cell4_fusion_degradation_runs.csv'))
COND = ['clean', 'noise_0.5sd', 'noise_1.0sd', 'noise_2.0sd', 'network_missing']
ARMS = ['network_only', 'power_only', 'early_fusion', 'late_uniform', 'late_val_weighted',
        'stacking_lr', 'late_calibrated', 'gate_maxconf', 'late_available_PH']
out = []
for c in COND:
    host = R[(R.condition == c) & (R.arm == 'host_only')].set_index(['pair_seed', 'model_seed'])['macro_f1']
    for a in ARMS:
        s = R[(R.condition == c) & (R.arm == a)]
        if s.empty: continue
        d = (s.set_index(['pair_seed', 'model_seed'])['macro_f1'] - host).values
        # unit = pairing seed (average over model seeds) -> 5 units, conservative
        du = (s.set_index(['pair_seed', 'model_seed'])['macro_f1'] - host).groupby(level=0).mean().values
        m, se = du.mean(), du.std(ddof=1) / np.sqrt(len(du))
        lo, hi = stats.t.interval(0.95, len(du) - 1, loc=m, scale=se) if se > 0 else (m, m)
        p = stats.ttest_1samp(du, 0).pvalue if se > 0 else (0.0 if m != 0 else 1.0)
        out.append({'condition': c, 'arm': a, 'mean_diff_vs_host': m, 'ci95_lo': lo, 'ci95_hi': hi,
                    'p_raw': p, 'runs_better_than_host': int((d > 0).sum()), 'runs': len(d)})
A = pd.DataFrame(out)
# Holm within each condition
for c in COND:
    idx = A.index[A.condition == c]; order = A.loc[idx, 'p_raw'].sort_values().index; k = len(order); run = 0
    for r, i in enumerate(order):
        run = max(run, min(1.0, A.loc[i, 'p_raw'] * (k - r))); A.loc[i, 'p_holm'] = run
A.to_csv(os.path.join(REV_DIR, 'R1_cell5A_degradation_stats.csv'), index=False)
print("="*70); print("(A) ARM minus HOST-ONLY, unit = pairing seed (n=5), Holm within condition"); print("="*70)
print(A.round(5).to_string(index=False))

summ = R.pivot_table(index='arm', columns='condition', values='macro_f1', aggfunc='mean')[COND]
summ['mean_over_conditions'] = summ.mean(1); summ['worst_case'] = summ[COND].min(1)
print("\nAcross-condition summary (mean macro-F1):"); print(summ.sort_values('worst_case', ascending=False).round(4))
summ.to_csv(os.path.join(REV_DIR, 'R1_cell5A_condition_summary.csv'))

# ======================= (B) Grouped feature removal =======================
LABELS = ['Benign', 'Recon', 'DoS']; lab = {c: i for i, c in enumerate(LABELS)}
NET_META = ['device', 'state', 'attack_raw', 'AttackGroup', 'source_file', 'n_flows_in_window']
def load(part):
    df = pd.read_csv(os.path.join(SPLITS, f'network_{part}.csv'), low_memory=False)
    cols = [c for c in df.columns if c not in NET_META and not c.endswith('_std')]
    cols = df[cols].select_dtypes(include=[np.number]).columns.tolist()
    return df[cols].apply(pd.to_numeric, errors='coerce').fillna(0), df['AttackGroup'].map(lab).values
Xtr_df, ytr = load('train'); Xte_df, yte = load('test')
cols = list(Xtr_df.columns)

# unique training vectors (label is consistent per vector - verified in Cell 2)
key_tr = pd.Series(list(map(tuple, np.round(Xtr_df.values.astype(float), 6))))
keep = ~key_tr.duplicated().values
Xtr_u, ytr_u = Xtr_df.values[keep], ytr[keep]
key_te = pd.Series(list(map(tuple, np.round(Xte_df.values.astype(float), 6))))
nd = ~key_te.isin(set(key_tr)).values
Xte_nd, yte_nd = Xte_df.values[nd], yte[nd]
print(f"\n(B) Unique train vectors: {len(ytr_u):,} | duplicate-free test rows: {len(yte_nd):,}")

base_name = lambda c: re.sub(r'_(mean|max|min|sum|median)$', '', c)
groups = {}
for i, c in enumerate(cols): groups.setdefault(base_name(c), []).append(i)
print(f"{len(cols)} features -> {len(groups)} groups:")
for g, ix in groups.items(): print(f"  {g}: {[cols[i] for i in ix]}")

SEEDS = [42, 123, 2026, 7, 99]
def score(feat_idx, seed):
    m = RandomForestClassifier(n_estimators=200, max_depth=20, class_weight='balanced',
                               random_state=seed, n_jobs=-1).fit(Xtr_u[:, feat_idx], ytr_u)
    p = m.predict(Xte_nd[:, feat_idx]); b = yte_nd != 0
    imp = m.feature_importances_
    return (f1_score(yte_nd, p, average='macro', labels=[0, 1, 2]),
            f1_score(yte_nd[b], p[b], average='macro', labels=[1, 2]), imp)

t0 = time.time(); all_idx = list(range(len(cols)))
base = [score(all_idx, s)[:2] for s in SEEDS]
base3, baseRD = np.mean([b[0] for b in base]), np.mean([b[1] for b in base])
print(f"\nBaseline (all features): 3-class {base3:.6f} | Recon/DoS {baseRD:.6f} ({time.time()-t0:.0f}s)")

# Leave-one-group-out
logo = []
for g, ix in groups.items():
    feats = [i for i in all_idx if i not in ix]
    sc = [score(feats, s)[:2] for s in SEEDS]
    logo.append({'removed_group': g, 'n_feats_removed': len(ix),
                 'f1_3class': np.mean([x[0] for x in sc]), 'f1_reconDoS': np.mean([x[1] for x in sc]),
                 'drop_3class': base3 - np.mean([x[0] for x in sc])})
    print(f"  LOGO -{g:28s} 3-class {logo[-1]['f1_3class']:.6f} ({time.time()-t0:.0f}s)")
L = pd.DataFrame(logo).sort_values('drop_3class', ascending=False)
L.to_csv(os.path.join(REV_DIR, 'R1_cell5B_leave_one_group_out.csv'), index=False)

# Greedy cumulative removal: drop the most important remaining GROUP (impurity importance summed per group)
remaining = dict(groups); steps = []
while len(remaining) > 1:
    feats = sorted(i for ix in remaining.values() for i in ix)
    res = [score(feats, s) for s in SEEDS]
    imp = np.mean([r[2] for r in res], axis=0); pos = {f: k for k, f in enumerate(feats)}
    gimp = {g: sum(imp[pos[i]] for i in ix) for g, ix in remaining.items()}
    top = max(gimp, key=gimp.get)
    steps.append({'groups_remaining': len(remaining), 'n_features': len(feats),
                  'f1_3class': np.mean([r[0] for r in res]), 'f1_reconDoS': np.mean([r[1] for r in res]),
                  'next_removed': top, 'its_importance': gimp[top]})
    print(f"  CUM {len(remaining):2d} groups ({len(feats):2d} feats): 3-class {steps[-1]['f1_3class']:.6f} "
          f"| Recon/DoS {steps[-1]['f1_reconDoS']:.6f} -> remove '{top}' ({time.time()-t0:.0f}s)")
    del remaining[top]
Cm = pd.DataFrame(steps); Cm.to_csv(os.path.join(REV_DIR, 'R1_cell5B_cumulative_removal.csv'), index=False)

print("\n" + "="*70); print("(B) LEAVE-ONE-GROUP-OUT (sorted by drop)"); print("="*70); print(L.round(6).to_string(index=False))
print("\n" + "="*70); print("(B) GREEDY CUMULATIVE REMOVAL"); print("="*70); print(Cm.round(6).to_string(index=False))
print(f"\nTotal time: {(time.time()-t0)/60:.1f} min | Saved to {REV_DIR}")

(A) ARM minus HOST-ONLY, unit = pairing seed (n=5), Holm within condition
      condition               arm  mean_diff_vs_host  ci95_lo  ci95_hi   p_raw  runs_better_than_host  runs  p_holm
          clean      network_only            0.01192  0.01098  0.01285 0.00000                     25    25 0.00003
          clean        power_only           -0.28313 -0.30867 -0.25759 0.00001                      0    25 0.00003
          clean      early_fusion            0.01225  0.01225  0.01225 0.00000                     25    25 0.00000
          clean      late_uniform            0.01044  0.00862  0.01227 0.00009                     25    25 0.00028
          clean late_val_weighted            0.01192  0.01098  0.01285 0.00000                     25    25 0.00003
          clean       stacking_lr            0.01192  0.01098  0.01285 0.00000                     25    25 0.00003
          clean   late_calibrated            0.00957  0.00787  0.01128 0.00010                     25    25 0.0002

In [ ]:
# Revision R1 - Cell 6: (1) Cross-device false-alarm rate + few-shot target benign (Reviewer 2 pt 8)
#                       (2) Leave-one-benign-file-out validation (Reviewer 3 pt 2, Reviewer 2 pt 5)
# 5 model seeds; Wilson 95% intervals for every rate computed on a small benign count.

import os, time
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from statsmodels.stats.proportion import proportion_confint

BASE = '/content/drive/MyDrive/Research/EVSE_Fusion_Project'
REV_DIR = os.path.join(BASE, 'For Journal', 'revision_R1')
LABELS = ['Benign', 'Recon', 'DoS']; lab = {c: i for i, c in enumerate(LABELS)}
SEEDS = [42, 123, 2026, 7, 99]
META = ['device', 'state', 'attack_raw', 'AttackGroup', 'source_file', 'n_flows_in_window']
wil = lambda k, n: proportion_confint(k, n, alpha=0.05, method='wilson') if n > 0 else (np.nan, np.nan)
def rf(s): return RandomForestClassifier(n_estimators=200, max_depth=20, class_weight='balanced',
                                         random_state=s, n_jobs=-1)

net = pd.read_csv(os.path.join(BASE, 'data/processed/network_agg.csv'), low_memory=False)
net = net[net['AttackGroup'].isin(LABELS)].reset_index(drop=True)
fcols = [c for c in net.columns if c not in META and not c.endswith('_std')]
fcols = net[fcols].select_dtypes(include=[np.number]).columns.tolist()
X = net[fcols].apply(pd.to_numeric, errors='coerce').fillna(0).values
y = net['AttackGroup'].map(lab).values
dev, src = net['device'].values, net['source_file'].values
print("Benign rows per device / file:")
print(net[net.AttackGroup == 'Benign'].groupby(['device', 'source_file']).size())

t0 = time.time()
# ====================== (1) Cross-device FAR + few-shot target benign ======================
A, Bd = dev == 'EVSE-A', dev == 'EVSE-B'
benA = np.where(A & (y == 0))[0]; attA = np.where(A & (y != 0))[0]; trB = np.where(Bd)[0]
print(f"\nEVSE-A benign rows: {len(benA)} | EVSE-A attack rows: {len(attA)} | EVSE-B rows (train): {len(trB)}")
rows1 = []
for k in [0, 5, 10, 20]:
    for s in SEEDS:
        r = np.random.default_rng(s)
        shots = r.choice(benA, size=k, replace=False) if k > 0 else np.array([], int)
        test_ben = np.setdiff1d(benA, shots)
        tr = np.concatenate([trB, shots])
        m = rf(s).fit(X[tr], y[tr])
        pb = m.predict(X[test_ben]); pa = m.predict(X[attA])
        fa = int((pb != 0).sum())
        lo, hi = wil(fa, len(test_ben))
        rows1.append({'k_target_benign': k, 'seed': s, 'n_test_benign': len(test_ben),
                      'false_alarms': fa, 'FAR': fa / len(test_ben), 'FAR_lo95': lo, 'FAR_hi95': hi,
                      'attack_detection_recall': float((pa != 0).mean()),
                      'reconDoS_macro_f1_on_A': f1_score(y[attA], pa, average='macro', labels=[1, 2])})
    print(f"  k={k:2d} done ({time.time()-t0:.0f}s)")
F = pd.DataFrame(rows1); F.to_csv(os.path.join(REV_DIR, 'R1_cell6_crossdevice_FAR.csv'), index=False)
print("\n" + "="*70); print("(1) TRAIN ON EVSE-B (+k EVSE-A benign shots) -> TEST ON EVSE-A"); print("="*70)
print(F.groupby('k_target_benign')[['n_test_benign', 'FAR', 'FAR_lo95', 'FAR_hi95',
                                    'attack_detection_recall', 'reconDoS_macro_f1_on_A']].mean().round(4))
print("(A->B FAR is untestable: EVSE-B contains no benign traffic.)")

# ====================== (2) Leave-one-benign-file-out ======================
tr_split = pd.read_csv(os.path.join(BASE, 'data/processed/splits/network_train.csv'),
                       usecols=['source_file', 'AttackGroup'], low_memory=False)
te_split = pd.read_csv(os.path.join(BASE, 'data/processed/splits/network_test.csv'),
                       usecols=['source_file', 'AttackGroup'], low_memory=False)
att_train_files = set(tr_split.loc[tr_split.AttackGroup != 'Benign', 'source_file'])
att_test_files = set(te_split.loc[te_split.AttackGroup != 'Benign', 'source_file'])
att_tr = np.where((y != 0) & np.isin(src, list(att_train_files)))[0]
att_te = np.where((y != 0) & np.isin(src, list(att_test_files)))[0]
ben_files = sorted(set(src[y == 0]))
print(f"\nBenign files: {ben_files}")
print(f"Attack rows: train-files {len(att_tr):,} | test-files {len(att_te):,} (paper's file-grouped split)")

rows2 = []
for f_tr in ben_files:
    f_te = [f for f in ben_files if f != f_tr]
    btr = np.where((y == 0) & (src == f_tr))[0]; bte = np.where((y == 0) & np.isin(src, f_te))[0]
    for s in SEEDS:
        m = rf(s).fit(X[np.concatenate([att_tr, btr])], y[np.concatenate([att_tr, btr])])
        te = np.concatenate([att_te, bte]); p = m.predict(X[te]); yt = y[te]
        pb = m.predict(X[bte]); correct = int((pb == 0).sum())
        lo, hi = wil(correct, len(bte))
        rows2.append({'benign_train_file': f_tr, 'benign_test_files': '|'.join(f_te), 'seed': s,
                      'n_benign_train': len(btr), 'n_benign_test': len(bte),
                      'benign_recall': correct / len(bte), 'recall_lo95': lo, 'recall_hi95': hi,
                      'FAR': 1 - correct / len(bte),
                      'f1_3class': f1_score(yt, p, average='macro', labels=[0, 1, 2]),
                      'f1_reconDoS': f1_score(yt[yt != 0], p[yt != 0], average='macro', labels=[1, 2])})
    print(f"  benign train file = {f_tr} done ({time.time()-t0:.0f}s)")
L = pd.DataFrame(rows2); L.to_csv(os.path.join(REV_DIR, 'R1_cell6_leave_one_benign_file_out.csv'), index=False)
print("\n" + "="*70); print("(2) LEAVE-ONE-BENIGN-FILE-OUT"); print("="*70)
print(L.groupby('benign_train_file')[['n_benign_train', 'n_benign_test', 'benign_recall', 'recall_lo95',
                                       'recall_hi95', 'FAR', 'f1_3class', 'f1_reconDoS']].mean().round(4).to_string())

# Reference: the paper's row-level benign split (13 test rows, 0 errors) as a Wilson interval
lo, hi = wil(13, 13)
print(f"\nReference - paper's row-split benign: 13/13 correct -> recall 95% Wilson CI [{lo:.3f}, {hi:.3f}]")
print(f"\nTotal time: {(time.time()-t0)/60:.1f} min | Saved to {REV_DIR}")

Benign rows per device / file:
device  source_file               
EVSE-A  EVSE-A-Charging-Benign.csv    14
        EVSE-A-idle-benign.csv        68
dtype: int64

EVSE-A benign rows: 82 | EVSE-A attack rows: 54766 | EVSE-B rows (train): 219671
  k= 0 done (81s)
  k= 5 done (174s)
  k=10 done (264s)
  k=20 done (354s)

(1) TRAIN ON EVSE-B (+k EVSE-A benign shots) -> TEST ON EVSE-A
                 n_test_benign     FAR  FAR_lo95  FAR_hi95  \
k_target_benign                                              
0                         82.0  1.0000    0.9552    1.0000   
5                         77.0  0.0909    0.0462    0.1745   
10                        72.0  0.0556    0.0256    0.1306   
20                        62.0  0.0129    0.0036    0.0791   

                 attack_detection_recall  reconDoS_macro_f1_on_A  
k_target_benign                                                   
0                                    1.0                  0.9995  
5                                    1.0    

In [ ]:
# Revision R1 - Cell 7: "Equally accurate" tolerance for the deployment claim (Reviewer 3, 2nd pt 3)
# Reads saved results only - no retraining. Same pre-specified margin as Cell 3 (DELTA = 0.005).
import os, numpy as np, pandas as pd
from scipy import stats

BASE = '/content/drive/MyDrive/Research/EVSE_Fusion_Project'
EXP = os.path.join(BASE, 'For Journal', 'new_experiments')
REV_DIR = os.path.join(BASE, 'For Journal', 'revision_R1')
DELTA = 0.005

s = pd.read_csv(os.path.join(EXP, 'part1_single_modality_10seed.csv'))
out = []
for mod in ['power', 'host', 'network']:
    rf = s[(s.modality == mod) & (s.model == 'RandomForest')].set_index('seed')['test_macro_f1']
    xg = s[(s.modality == mod) & (s.model == 'XGBoost')].set_index('seed')['test_macro_f1']
    d = (rf - xg).dropna().values; n = len(d); m = d.mean(); se = d.std(ddof=1) / np.sqrt(n)
    if se == 0:
        out.append({'modality': mod, 'n_seeds': n, 'mean_RF': rf.mean(), 'mean_XGB': xg.mean(),
                    'mean_diff_RF_minus_XGB': m, 'ci90_lo': m, 'ci90_hi': m, 'p_tost': 0.0 if abs(m) < DELTA else 1.0})
        continue
    p_lo = 1 - stats.t.cdf((m + DELTA) / se, n - 1); p_hi = stats.t.cdf((m - DELTA) / se, n - 1)
    ci = stats.t.interval(0.90, n - 1, loc=m, scale=se)
    out.append({'modality': mod, 'n_seeds': n, 'mean_RF': rf.mean(), 'mean_XGB': xg.mean(),
                'mean_diff_RF_minus_XGB': m, 'ci90_lo': ci[0], 'ci90_hi': ci[1], 'p_tost': max(p_lo, p_hi)})
T = pd.DataFrame(out); T['equivalent_within_0.005'] = (T.ci90_lo > -DELTA) & (T.ci90_hi < DELTA) & (T.p_tost < 0.05)
print("RF vs XGBoost, 10 paired seeds, TOST margin +/-0.005"); print(T.round(6).to_string(index=False))

for name in ['inference_latency_model_size_CORRECTED.csv', 'inference_latency_model_size.csv']:
    p = os.path.join(EXP, name)
    if os.path.exists(p):
        z = pd.read_csv(p); print(f"\nDeployment table source: {name}"); break
pw = z[z.modality == 'power'].set_index('model')
print(pw[['test_macro_f1', 'latency_ms_per_sample', 'model_size_kb']].round(5))
print(f"\nPower RF / XGBoost on-disk size ratio: {pw.loc['RandomForest','model_size_kb'] / pw.loc['XGBoost','model_size_kb']:.2f}x")
print("Size = joblib.dump file size (seed-42 model); latency = 50 repeated full-test-set predictions / n_test, Colab CPU.")
T.to_csv(os.path.join(REV_DIR, 'R1_cell7_rf_vs_xgb_equivalence.csv'), index=False)

RF vs XGBoost, 10 paired seeds, TOST margin +/-0.005
modality  n_seeds  mean_RF  mean_XGB  mean_diff_RF_minus_XGB   ci90_lo   ci90_hi   p_tost  equivalent_within_0.005
   power       10 0.715150  0.715711               -0.000561 -0.001411  0.000289 0.000003                     True
    host       10 0.987455  0.986385                0.001071 -0.000000  0.002142 0.000043                     True
 network       10 0.999972  0.999983               -0.000010 -0.000017 -0.000003 0.000000                     True

Deployment table source: inference_latency_model_size_CORRECTED.csv
                    test_macro_f1  latency_ms_per_sample  model_size_kb
model                                                                  
LogisticRegression        0.53088                0.00018        0.98340
MLP                       0.69537                0.00018        9.80957
RandomForest              0.71695                0.04108   153091.11035
XGBoost                   0.71481                0.02642  